In [6]:
import datetime as dt
import xarray as xr
import fsspec
import s3fs
import os.path
import matplotlib.pyplot as plt
import pandas as pd

In [16]:
def get_geo_data(sat='goes-west', lyr=2020, idyjl=1, product='ABI-L2-SSTF'):
    # arguments
    # sat   goes-east,goes-west
    # lyr   year
    # idyjl day of year
    # Product. For a full list see: https://github.com/NOAA-Big-Data-Program/nodd-data-docs/tree/main/GOES

    d = dt.datetime(lyr,1,1) + dt.timedelta(days=idyjl)
    fs = s3fs.S3FileSystem(anon=True) #connect to s3 bucket!

    #create strings for the year and julian day
    imon,idym=d.month,d.day
    syr,sjdy,smon,sdym = str(lyr).zfill(4),str(idyjl).zfill(3),str(imon).zfill(2),str(idym).zfill(2)

    #use glob to list all the files in the directory
    if sat=='goes-east':
        file_location,var = fs.glob(f's3://noaa-goes16/{product}/{syr}/{sjdy}/*/*.nc'),'SST'
    if sat=='goes-west':
        file_location,var = fs.glob(f's3://noaa-goes17/{product}/{syr}/{sjdy}/*/*.nc'),'SST'

    #make a list of links to the file keys
    if len(file_location)<1:
        return file_ob
    file_ob = [fs.open(file) for file in file_location]        #open connection to files

    #open all the day's data
    ds = xr.open_mfdataset(file_ob,combine='nested',concat_dim='time') #note file is super messed up formatting

    #clean up coordinates which are a MESS in GOES
    #rename one of the coordinates that doesn't match a dim & should

    return ds

In [14]:
def date_to_doy(year, month, day):
    return dt.datetime(year, month, day).timetuple().tm_yday

def get_storm_window(sat, storm_date, days_before=7, days_after=5, product='ABI-L2-SSTF'):
    datasets = []
    
    start = storm_date - dt.timedelta(days=days_before)
    end   = storm_date + dt.timedelta(days=days_after)
    
    current = start
    while current <= end:
        lyr   = current.year
        idyjl = current.timetuple().tm_yday
        print(f"Fetching {current.strftime('%Y-%m-%d')} (DOY {idyjl})...")
        try:
            ds = get_geo_data(sat=sat, lyr=lyr, idyjl=idyjl, product=product)
            datasets.append(ds)
            print(f"  OK — {len(datasets)} days loaded so far")
        except Exception as e:
            print(f"  FAILED: {type(e).__name__}: {e}")
        current += dt.timedelta(days=1)
        

    print(f"\nTotal days fetched: {len(datasets)}")
    
    if not datasets:
        raise ValueError("No data fetched — check errors above.")

    full_ds = xr.concat(datasets, dim='time', combine_attrs='drop_conflicts')
    return full_ds

In [18]:
storms = {
    'Maria':  ('goes-east', dt.datetime(2017, 9, 20)),
    'Dorian': ('goes-east', dt.datetime(2019, 9,  1)),
    'Laura':  ('goes-east', dt.datetime(2020, 8, 27)),
    'Ida':    ('goes-east', dt.datetime(2021, 8, 29)),
    'Ian':    ('goes-east', dt.datetime(2022, 9, 28)),
    'Isaiah': ('goes-east', dt.datetime(2020, 7, 29)),
}

name = 'Ian'
sat, storm_date = storms[name]

ds_ida = get_storm_window(sat=sat, storm_date=storm_date)

Fetching 2022-09-21 (DOY 264)...
  OK — 1 days loaded so far
Fetching 2022-09-22 (DOY 265)...
  OK — 2 days loaded so far
Fetching 2022-09-23 (DOY 266)...
  OK — 3 days loaded so far
Fetching 2022-09-24 (DOY 267)...
  OK — 4 days loaded so far
Fetching 2022-09-25 (DOY 268)...
  OK — 5 days loaded so far
Fetching 2022-09-26 (DOY 269)...
  OK — 6 days loaded so far
Fetching 2022-09-27 (DOY 270)...
  OK — 7 days loaded so far
Fetching 2022-09-28 (DOY 271)...
  OK — 8 days loaded so far
Fetching 2022-09-29 (DOY 272)...
  OK — 9 days loaded so far
Fetching 2022-09-30 (DOY 273)...
  OK — 10 days loaded so far
Fetching 2022-10-01 (DOY 274)...
  OK — 11 days loaded so far
Fetching 2022-10-02 (DOY 275)...
  OK — 12 days loaded so far
Fetching 2022-10-03 (DOY 276)...
  OK — 13 days loaded so far

Total days fetched: 13


In [20]:
import json, numpy as np, os, tempfile

def export_frames(ds, out_dir='data'):
    os.makedirs(out_dir, exist_ok=True)
    
    sst = ds['SST']
    x_vals = ds['x'].values
    y_vals = ds['y'].values
    
    xx, yy = np.meshgrid(x_vals, y_vals)
    
    for i in range(len(sst.time)):
        if i % 2 != 0:
            continue
        frame = sst.isel(time=i).values
        timestamp = pd.Timestamp(sst.time.isel(time=i).values.item()).isoformat()

        valid = np.isfinite(frame)
        
        # keep every 10th valid pixel
        N = 25
        valid_idx = np.where(valid.ravel())[0][::N]
        valid2d = np.unravel_index(valid_idx, frame.shape)
        
        temps = frame[valid2d]
        xs    = xx[valid2d]
        ys    = yy[valid2d]
        
        if temps.mean() > 200:
            temps = temps - 273.15
        
        records = [
            {'x': round(float(x), 6),
             'y': round(float(y), 6),
             'temp': round(float(t), 2)}
            for x, y, t in zip(xs, ys, temps)
        ]
        
        output = {'time': timestamp, 'pixels': records}
        with open(f'{out_dir}/frame_{i}.json', 'w') as f:
            json.dump(output, f)
        
        print(f"frame_{i}.json — {len(records)} valid pixels")

In [22]:
export_frames(ds_ida, out_dir='Ian')

frame_0.json — 673604 valid pixels
frame_2.json — 673633 valid pixels
frame_4.json — 673599 valid pixels
frame_6.json — 673605 valid pixels
frame_8.json — 673601 valid pixels
frame_10.json — 673626 valid pixels
frame_12.json — 673625 valid pixels
frame_14.json — 673617 valid pixels
frame_16.json — 673586 valid pixels
frame_18.json — 673585 valid pixels
frame_20.json — 673567 valid pixels
frame_22.json — 673570 valid pixels
frame_24.json — 673561 valid pixels
frame_26.json — 673564 valid pixels
frame_28.json — 673598 valid pixels
frame_30.json — 673630 valid pixels
frame_32.json — 673625 valid pixels
frame_34.json — 673630 valid pixels
frame_36.json — 673642 valid pixels
frame_38.json — 673669 valid pixels
frame_40.json — 673665 valid pixels
frame_42.json — 673653 valid pixels
frame_44.json — 673645 valid pixels
frame_46.json — 673663 valid pixels
frame_48.json — 673683 valid pixels
frame_50.json — 673727 valid pixels
frame_52.json — 673724 valid pixels
frame_54.json — 673710 valid pixe